In [19]:
### follow this notebook fo rmultiple charge trap implementation

In [20]:
import time
import sys
import os

# Add the parent directory to sys.path
notebook_path = os.getcwd()  # Get the current working directory where the notebook is located
parent_directory = os.path.dirname(notebook_path)  # Move to the parent directory
sys.path.append(parent_directory)

# Now you can import config
from config import *
#from Minimal_Model_2D import *
from Minimal_Model_2D import *
import time

np.set_printoptions(threshold=sys.maxsize)

In [16]:
class Minimal_Model_multiple_charge_trap_potential(twoD_Minimal_Model):

    ### overriding a H same spin method to include multiple charge trap 

    def H_same_spin(self, x_ct= np.empty(5), y_ct=np.empty(5), z_ct=np.empty(5), z_im=np.empty(5), z_qw=0, dis_mag=0, uncorr_var=1, lmbda=0, seed=None, kx=0, ky=0):

        #print('a- same spin class overridden')

        M = (hbm/self.meff_sm* self.twodOP.KxSq(kx) + hbm/self.meff_sm* self.twodOP.KySq(ky) - self.chem_pot() + 1/4/hbm*self.meff_sm*self.omega**2*self.twodOP.ysq()  +self.dis_pot_matrix(dis_mag) +self.Disorder_full(uncorr_var, lmbda, seed) )

        for i in range(x_ct.shape[0]):

            #print(i)
            M = M + self.charge_pot_lattice(x_ct[i], y_ct[i], z_ct[i], z_im[i], z_qw)
 
        return M


In [21]:
NSM = 60            # number of sm x lattice points
M = 60              # should be atleast thrice confinement length
AX = 5              # lattice spacing in nm
AY = 5              # lattice spacing in nm
DELTA = 0.165       # in meV
MEFF_SC = 0.05    # proximitized Ge
MEFF_SM = 0.05     # Ge
NSC = 800
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
ALPHAT = 400        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 8           # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 100           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC

MY= 5 

print('lengths=', M*AY, LY)
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print('subband energies=', OMEGA/2, 3*OMEGA/2)

print('4t=', 4*hbm/AX**2/MEFF_SM)

MU_SC =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably 
MU_SM =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably

print('mu = ', MU_SM)


UNCORR_VAR = 0  ### uncorrelated variance of disorder
LMBDA = 0
SEED = 58  ## seed for the disorder

# Define the range and the size of the array
x_low = (NSC+1)*AX  # Start of the range (inclusive)
x_high = (NSC+NSM+1)*AX  # End of the range (exclusive)

y_low = 0.0  # Start of the range (inclusive)
y_high = M*AY  # End of the range (exclusive)

size = 5  # Size of the array

# Generate the array
X_CT = np.random.uniform(x_low, x_high, size)
Y_CT = np.random.uniform(y_low, y_high, size)
Z_CT = np.ones(size)*50                          ###all charg etraps assumed to be in oxide interface
Z_IM  = np.ones(size)*64 

print(X_CT)
print(Y_CT)
print(Z_CT)
print(Z_IM)


ModCT = Minimal_Model_multiple_charge_trap_potential(NTOT, NSM, NSC, M, AX, AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM)

lengths= 300 100
subband energies= 0.30505912614720065 0.915177378441602
4t= 122.02365045888027
mu =  0.3660709513766408
[4022.14110148 4286.65365448 4187.84378503 4302.19944609 4161.53863124]
[123.31073167 235.60094975  54.34695437 203.56226603  79.26695758]
[50. 50. 50. 50. 50.]
[64. 64. 64. 64. 64.]


In [ ]:

H = ModCT.H_sys(PHI, 'cu', X_CT, Y_CT, Z_CT, Z_IM)

E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

Es = np.sort(E)/DELTA

#Ep[i,:] = Es

print(Es)

#print(H.shape)

In [18]:
ITER = 21

K_NUM = 4

Ep = np.empty((ITER,K_NUM))

Z_CT = np.ones(size)*50                          ###all charg etraps assumed to be in oxide interface
Z_IM  = np.ones(size)*64 

for i in range(ITER):

    # Generate the array
    X_CT = np.random.uniform(x_low, x_high, size)
    Y_CT = np.random.uniform(y_low, y_high, size)  
        
    H = ModCT.H_sys(PHI, 'cu', X_CT, Y_CT, Z_CT, Z_IM)

    E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

    Es = np.sort(E)/DELTA

    Ep[i,:] = Es

    if i%5==0:

        print(Es)

    #print(H.shape)

    
np.save("cubic_spin_splitting_{}_charge_traps_{}_iter_const_z.npy".format(size,ITER),Ep)


[-0.9529461  -0.95294608  0.95294608  0.9529461 ]
[-0.92471083 -0.92470897  0.92470897  0.92471083]
[-0.9484576  -0.94845716  0.94845716  0.9484576 ]
[-0.84677562 -0.84677534  0.84677534  0.84677562]
[-0.95060407 -0.95060349  0.95060349  0.95060407]


In [22]:
Ep = np.load("cubic_spin_splitting_{}_charge_traps_{}_iter_const_z.npy".format(size,ITER))

In [24]:
K_NUM = Ep.shape[1]

In [25]:
Ep_splitting = Ep[:,K_NUM//2+1] - Ep[:, K_NUM//2]

In [26]:
Ep_splitting.shape

(21,)

In [27]:
np.mean(Ep_splitting)

2.8682379345263646e-06

In [28]:
np.var(Ep_splitting)

3.0792185151419914e-11

In [30]:
print(Ep_splitting)

[1.68424849e-08 9.22118683e-07 2.12326243e-07 2.92155890e-07
 7.56883171e-08 1.86405395e-06 2.12516981e-05 1.36689684e-07
 1.06116534e-07 3.42191873e-06 4.42829642e-07 4.49930801e-06
 1.48871762e-06 2.68141141e-07 3.05502060e-07 2.74286889e-07
 1.66586041e-05 8.10872892e-08 7.12499599e-06 2.03870852e-07
 5.86044435e-07]


In [11]:
print(Ep)

[[-0.13192261 -0.1317863   0.1317863   0.13192261]
 [-0.09159026 -0.09129954  0.09129954  0.09159026]
 [-0.14807051 -0.14791455  0.14791455  0.14807051]
 [-0.31509968 -0.3150037   0.3150037   0.31509968]
 [-0.23204089 -0.23197075  0.23197075  0.23204089]
 [-0.16820595 -0.16808997  0.16808997  0.16820595]
 [-0.18556864 -0.18541731  0.18541731  0.18556864]
 [-0.3538026  -0.35372954  0.35372954  0.3538026 ]
 [-0.40178512 -0.40172177  0.40172177  0.40178512]
 [-0.33471526 -0.33465444  0.33465444  0.33471526]
 [-0.17511356 -0.17498583  0.17498583  0.17511356]
 [-0.33155985 -0.3314515   0.3314515   0.33155985]
 [-0.22481756 -0.2247137   0.2247137   0.22481756]
 [-0.10197375 -0.10180226  0.10180226  0.10197375]
 [-0.09048275 -0.09024585  0.09024585  0.09048275]
 [-0.09795129 -0.09773942  0.09773942  0.09795129]
 [-0.23976663 -0.23964305  0.23964305  0.23976663]
 [-0.14867137 -0.14856598  0.14856598  0.14867137]
 [-0.28077385 -0.2806741   0.2806741   0.28077385]
 [-0.09238114 -0.09221115  0.09